In [2]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


Matplotlib is building the font cache; this may take a moment.


In [11]:
from pathlib import Path

# 1. Define project paths and settings

try:
    PROJECT_DIR = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_DIR = Path.cwd().parent

INPUT_FILE = (
    PROJECT_DIR / "data" / "processed" / "clean_stocks_data.csv"
)
REPORT_DIR = PROJECT_DIR / "reports"
FIGURE_DIR = PROJECT_DIR / "figures"

# Ensure all target folders exist
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

INITIAL_INVESTMENT = 10_000



In [13]:
# 2. Load and validate the clean dataset

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"File not found: {INPUT_FILE}\n"
        "Complete Step 3 before running this script."
    )

df = pd.read_csv(INPUT_FILE, parse_dates=["Date"])

required_columns = {"Date", "Close", "Ticker"}
if not required_columns.issubset(df.columns):
    raise ValueError(
        f"Dataset must contain: {sorted(required_columns)}"
    )

df = df.dropna(subset=["Date", "Close", "Ticker"]).copy()
df["Ticker"] = df["Ticker"].str.upper().str.strip()

if df.duplicated(["Date", "Ticker"]).any():
    raise ValueError("Duplicate ticker-date records found.")

if (df["Close"] <= 0).any():
    raise ValueError("Non-positive closing prices found.")

# Success confirmation
print(f"✅ Dataset validation successful!")
print(f"Total valid records loaded: {len(df):,}")
print(f"Unique tickers found: {df['Ticker'].unique()}")
print(f"Date range covered: {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}")


✅ Dataset validation successful!
Total valid records loaded: 3,762
Unique tickers found: <StringArray>
['AAPL', 'MSFT', 'NVDA']
Length: 3, dtype: str
Date range covered: 2021-10-22 to 2026-10-21


In [14]:
# 3. Reshape data: one column per stock

prices = df.pivot(
    index="Date",
    columns="Ticker",
    values="Close"
).sort_index()

expected_tickers = ["AAPL", "MSFT", "NVDA"]
missing_tickers = [
    ticker for ticker in expected_tickers
    if ticker not in prices.columns
]

if missing_tickers:
    raise ValueError(
        f"Missing stock data for: {missing_tickers}"
    )

prices = prices[expected_tickers]

# Use dates on which all three stocks have prices.
# This ensures a fair comparison over the same period.
prices = prices.dropna(how="any")

if len(prices) < 2:
    raise ValueError(
        "Not enough common dates to calculate returns."
    )


In [15]:
# 4. Calculate daily returns

daily_returns = prices.pct_change(fill_method=None)

daily_returns = daily_returns.iloc[1:].copy()

daily_returns.index.name = "Date"

daily_returns.to_csv(
    REPORT_DIR / "daily_stocks_returns.csv"
)


In [16]:
# 5. Calculate cumulative returns and investment growth
# Each stock starts at the same $10,000 investment.

investment_values = (
    prices / prices.iloc[0]
) * INITIAL_INVESTMENT

investment_values.index.name = "Date"

investment_values.to_csv(
    REPORT_DIR / "investment_growth.csv"
)

cumulative_returns = (
    prices / prices.iloc[0] - 1
) * 100

cumulative_returns.index.name = "Date"

cumulative_returns.to_csv(
    REPORT_DIR / "cumulative_returns.csv"
)



In [17]:
# 6. Build the performance summary

start_date = prices.index[0]
end_date = prices.index[-1]

calendar_days = (end_date - start_date).days

if calendar_days <= 0:
    raise ValueError("Invalid analysis date range.")

summary_rows = []

for ticker in expected_tickers:
    start_price = prices[ticker].iloc[0]
    end_price = prices[ticker].iloc[-1]

    total_return = (end_price / start_price) - 1

    ending_value = investment_values[ticker].iloc[-1]

    # Compound annual growth rate (CAGR)
    annualized_return = (
        (end_price / start_price) ** (365.25 / calendar_days)
        - 1
    )

    summary_rows.append({
        "Ticker": ticker,
        "Start_Date": start_date.date(),
        "End_Date": end_date.date(),
        "Trading_Days": len(prices),
        "Starting_Adjusted_Price": start_price,
        "Ending_Adjusted_Price": end_price,
        "Total_Return_Pct": total_return * 100,
        "Annualized_Return_Pct": annualized_return * 100,
        "Initial_Investment_USD": INITIAL_INVESTMENT,
        "Ending_Investment_Value_USD": ending_value,
        "Profit_Loss_USD": ending_value - INITIAL_INVESTMENT,
    })

summary = pd.DataFrame(summary_rows)

summary = summary.sort_values(
    "Total_Return_Pct", ascending=False
)

summary.to_csv(
    REPORT_DIR / "performance_summary.csv",
    index=False
)


In [18]:
# 7. Chart 1: normalized stock price performance

normalized_prices = (
    prices / prices.iloc[0]
) * 100

plt.figure(figsize=(12, 6))

for ticker in expected_tickers:
    plt.plot(
        normalized_prices.index,
        normalized_prices[ticker],
        label=ticker,
        linewidth=1.8
    )

plt.title("Stock Performance Comparison (Starting Value = 100)")
plt.xlabel("Date")
plt.ylabel("Normalized Adjusted Price")
plt.legend(title="Stock")
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "stock_price_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()



In [19]:
# 8. Chart 2: hypothetical investment growth

plt.figure(figsize=(12, 6))

for ticker in expected_tickers:
    plt.plot(
        investment_values.index,
        investment_values[ticker],
        label=ticker,
        linewidth=1.8
    )

plt.axhline(
    INITIAL_INVESTMENT,
    linestyle="--",
    linewidth=1,
    label="Initial investment"
)

plt.title("Growth of a Hypothetical $10,000 Investment")
plt.xlabel("Date")
plt.ylabel("Portfolio Value (USD)")
plt.legend(title="Stock")
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "investment_growth.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()


In [22]:
# 9. Print findings

print("=" * 65)
print("STOCK MARKET RETURN ANALYSIS")
print("=" * 65)

print(f"Analysis start: {start_date.date()}")
print(f"Analysis end:   {end_date.date()}")
print(f"Common trading dates: {len(prices):,}")
print(f"Initial investment per stock: ${INITIAL_INVESTMENT:,.2f}")

print("\nPERFORMANCE SUMMARY")
print(
    summary.round(2).to_string(index=False)
)

print("\nOUTPUT FILES")
print(" - reports/daily_stocks_returns.csv")
print(" - reports/cumulative_returns.csv")
print(" - reports/investment_growth.csv")
print(" - reports/performance_summary.csv")
print(" - figures/stock_price_comparison.png")
print(" - figures/investment_growth.png")

print("\nAnalysis completed successfully.")


STOCK MARKET RETURN ANALYSIS
Analysis start: 2021-10-22
Analysis end:   2026-10-21
Common trading dates: 1,254
Initial investment per stock: $10,000.00

PERFORMANCE SUMMARY
Ticker Start_Date   End_Date  Trading_Days  Starting_Adjusted_Price  Ending_Adjusted_Price  Total_Return_Pct  Annualized_Return_Pct  Initial_Investment_USD  Ending_Investment_Value_USD  Profit_Loss_USD
  NVDA 2021-10-22 2026-10-21          1254                    22.63                 221.97            881.02                  57.93                   10000                     98102.47         88102.47
  AAPL 2021-10-22 2026-10-21          1254                   145.03                 343.99            137.19                  18.87                   10000                     23718.88         13718.88
  MSFT 2021-10-22 2026-10-21          1254                   296.71                 532.95             79.62                  12.44                   10000                     17962.03          7962.03

OUTPUT FILES
 - re